# Human-in-the-Loop (HITL) in LangGraph

Human-in-the-Loop (HITL) allows a workflow to pause execution at specific breakpoints, wait for human review/input/approval, and resume execution with or without state updates.

### Core Concepts:
1. **Checkpointer (`InMemorySaver`)**: Preserves state across graph pauses.
2. **Breakpoints (`interrupt_before` / `interrupt_after`)**: Designates nodes where execution should halt.
3. **State Inspection (`graph.get_state(config)`)**: Inspects current node outputs and the next pending node.
4. **State Modification (`graph.update_state(config, values)`)**: Injects human review, edits, or feedback.
5. **Resuming Execution (`graph.invoke(None, config)`)**: Continues execution from the interrupted state.

### 1. Imports and Setup

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

### 2. Define State
Define the schema for data passed between nodes, including draft output, human feedback, and approval status.

In [ ]:
class ContentState(TypedDict):
    topic: str
    draft: str
    feedback: Optional[str]
    approved: bool
    final_content: str

### 3. Define Nodes
- **`generate_draft`**: Creates an initial draft.
- **`finalize_content`**: Reviews and finalizes the output incorporating human feedback.

In [ ]:
def generate_draft(state: ContentState):
    topic = state["topic"]
    print(f"\n[Node: generate_draft] Creating draft for: '{topic}'...")
    draft_text = f"AI in {topic} is accelerating progress with automated workflows and intelligent assistance."
    return {"draft": draft_text, "approved": False}

def finalize_content(state: ContentState):
    print("\n[Node: finalize_content] Finalizing content with human feedback...")
    draft = state["draft"]
    feedback = state.get("feedback")
    
    if feedback:
        final = f"{draft}\n\n[Human Feedback Incorporated]: {feedback}"
    else:
        final = f"{draft}\n\n[Status]: Approved without modifications."
        
    return {"final_content": final, "approved": True}

### 4. Build and Compile Graph with Breakpoints
We set `interrupt_before=["finalize_content"]` so the graph automatically pauses before finalizing.

In [ ]:
builder = StateGraph(ContentState)

# Add nodes
builder.add_node("generate_draft", generate_draft)
builder.add_node("finalize_content", finalize_content)

# Add edges
builder.add_edge(START, "generate_draft")
builder.add_edge("generate_draft", "finalize_content")
builder.add_edge("finalize_content", END)

# Checkpointer is required for HITL to save checkpointed state across pauses
checkpointer = InMemorySaver()

# Compile with interrupt before finalize_content
graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["finalize_content"]
)

print("Workflow compiled with HITL interrupt before 'finalize_content'!")

### 5. Run Workflow: Step 1 (Trigger & Pause)
Invoke with a unique `thread_id`. Execution runs up to `finalize_content` and halts.

In [ ]:
config = {"configurable": {"thread_id": "hitl_thread_1"}}

initial_input = {"topic": "Healthcare"}
graph.invoke(initial_input, config=config)

### 6. Inspect Paused State (Human Review)
Inspect the generated draft and verify what node is waiting to be executed.

In [ ]:
current_state = graph.get_state(config)

print("--- Current Paused State ---")
print("Values:", current_state.values)
print("Draft content:", current_state.values.get("draft"))
print("Next node pending execution:", current_state.next)

### 7. Human Action: Provide Feedback (Update State)
The human injects feedback or updates the draft using `update_state`.

In [ ]:
human_feedback = "Ensure patient data privacy, security, and HIPAA compliance are highlighted."

graph.update_state(
    config,
    {"feedback": human_feedback}
)

print("State updated with human feedback!")

### 8. Resume Workflow: Step 2
Pass `None` to `graph.invoke(None, config=config)` to resume execution.

In [ ]:
final_state = graph.invoke(None, config=config)

print("\n--- Final Result after Resuming ---")
print(final_state["final_content"])